# Trinet IMU ↔ Camera Time Sync — end-to-end example

Align a Trinet camera's IMU with its video frames, from a raw `.mp4` to
**per-frame interpolated IMU**. **Replace `VIDEO` in the next cell with your own
recording and run all cells.**

Pipeline: extract sidecars → put frames on the IMU clock (`sof_timestamp_ns`)
→ apply the Kalibr `timeshift_cam_imu` → interpolate the IMU at each frame.

Theory and the per-version guide: [`docs/imu_video_sync.md`](../docs/imu_video_sync.md).


In [ ]:
from pathlib import Path
import sys, numpy as np

# Make the `trinet_tools` package importable without a pip install: walk up from
# the working dir to the repo root (the folder that contains trinet_tools/).
# Alternatively run `pip install -e .` from the repo root and drop this loop.
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "trinet_tools" / "__init__.py").exists():
        sys.path.insert(0, str(_p)); break

VIDEO       = "your_recording.mp4"   # <-- replace with your Trinet .mp4
CALIBRATION = None                    # <-- optional: path to this unit's calibration.json
OUT         = Path("out")            # where sidecars are written

## 1. Extract the IMU + per-frame timestamps from the video

`extract_sei` walks the H.264 SEI and writes `imu.bin` + `frames.bin` (+ a clean
`video.mp4`). It prints the format `version` — that tells you which camera
generation produced the recording (see the doc's version table).


In [ ]:
from trinet_tools.extract_sei import extract
extract(Path(VIDEO), OUT)            # -> out/imu.bin, out/frames.bin, out/video.mp4

## 2. Load and check the per-frame timing

Align the IMU to each frame's hardware `sof_timestamp_ns` — **never** the video
PTS (that carries delivery latency). The nearest IMU sample to each frame should
be a fraction of the IMU period away.


In [ ]:
from trinet_tools.reader import read_imu, read_vts
imu = read_imu(OUT / "imu.bin")
vts = read_vts(OUT / "frames.bin")
print(f".imu v{imu.header.version} ({imu.header.generation}), .vts v{vts.header.version}, "
      f"mid_exposure={vts.is_mid_exposure}")

sof    = vts.sof_timestamps_ns.astype(np.int64)   # per-frame capture time (ns, camera clock)
imu_ts = imu.timestamps_ns.astype(np.int64)
assert sof.max() > 0, (
    "sof is all-zero: a UVC recording from an SEI-v5 camera has no per-frame "
    "hardware timestamp in the stream. Use the device's on-board .vts, or update "
    "the camera firmware (SEI v6). See docs/imu_video_sync.md.")

d = np.diff(sof) / 1e6                             # frame-to-frame ms
print(f"cadence: {d.mean():.3f} ms ({1000/d.mean():.2f} fps), std {d.std():.3f} ms")
idx = np.clip(np.searchsorted(imu_ts, sof), 1, len(imu_ts) - 1)
near_us = np.minimum(np.abs(imu_ts[idx]-sof), np.abs(imu_ts[idx-1]-sof)) / 1e3
print(f"frame -> nearest IMU sample: median {np.median(near_us):.0f} us, "
      f"95th {np.percentile(near_us,95):.0f} us")

## 3. Apply the Kalibr time offset (`timeshift_cam_imu`)

`sof_timestamp_ns` gives sub-ms *precision*, but a constant offset remains
(rolling-shutter readout centre + IMU group delay + pipeline). Kalibr calibrates
it; supply this unit's `calibration.json` to apply it. The convention is
`t_imu = t_cam + timeshift_cam_imu_sec`.


In [ ]:
import json
timeshift_s = 0.0
if CALIBRATION:
    timeshift_s = json.load(open(CALIBRATION))["extrinsics"]["timeshift_cam_imu_sec"]
    print(f"timeshift_cam_imu = {timeshift_s*1e3:.2f} ms (from calibration.json)")
else:
    print("no calibration.json -> timeshift 0; supply one for a physically-correct offset")

frame_t_imu = sof + int(round(timeshift_s * 1e9))   # IMU-clock time of each frame

## 4. Interpolate the IMU at each frame's capture instant

`accel[i]` / `gyro[i]` are the inertial readings at the moment frame `i` was
captured — ready to feed a fusion / VIO front-end alongside the frame.


In [ ]:
def interp_at(t, channel):              # linear interpolation of one IMU axis at times t
    return np.interp(t, imu_ts, channel)

accel = np.stack([interp_at(frame_t_imu, imu.accel[:, k]) for k in range(3)], axis=1)
gyro  = np.stack([interp_at(frame_t_imu, imu.gyro[:,  k]) for k in range(3)], axis=1)
print("per-frame accel/gyro:", accel.shape, gyro.shape)

## 5. Visualise — do the frames land on the IMU signal?

Overlay the per-frame interpolated gyro magnitude on the raw IMU gyro trace. The
red points should sit on the blue curve if the timing is right.


In [ ]:
import matplotlib.pyplot as plt   # pip install matplotlib if needed
gmag = np.linalg.norm(imu.gyro, axis=1)
t0 = imu_ts[0]
ti = (imu_ts - t0) / 1e9
tf = (frame_t_imu - t0) / 1e9
lo, hi = 5.0, 8.0                       # a 3-second window
w  = (ti >= lo) & (ti <= hi)
fw = (tf >= lo) & (tf <= hi)
plt.figure(figsize=(11, 3))
plt.plot(ti[w], gmag[w], lw=0.8, label="IMU |gyro|")
plt.scatter(tf[fw], np.linalg.norm(gyro[fw], axis=1), s=12, c="r", zorder=3,
            label="per-frame (interpolated)")
plt.xlabel("time (s)"); plt.ylabel("|gyro| (rad/s)")
plt.title("Frames placed on the IMU timeline"); plt.legend(); plt.tight_layout(); plt.show()

### Notes

- **Which version do I have?** `extract_sei` prints `version=N`, or read
  `imu.header.version` / `vts.header.version`. The per-version sync table is in
  [`docs/imu_video_sync.md`](../docs/imu_video_sync.md).
- On a representative v6 recording: ~30 fps cadence, frame→nearest-IMU median
  ~0.8 ms (bounded by the IMU period), `timeshift_cam_imu` ≈ −16 ms.
- For per-row rolling-shutter precision use
  `line_delay = vts.readout_time_us / image_height` (the Kalibr line delay).
- Always apply **your recording's own** `timeshift_cam_imu` — it is a stable
  per-design constant but is measured per calibration.
